In [ ]:
# Standard library imports
import os
import sys
import re
import logging
import warnings

# Third-party imports
import optuna
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from xgboost import XGBRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# Local imports
module_path = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if module_path not in sys.path:
    sys.path.append(module_path)
from paths import BASE_INPUT_PATH, BASE_OUTPUT_PATH

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

In [ ]:
def make_xgb_safe_columns(df):
    """
    XGBoost requires feature names to be strings and disallows [, ], <.
    """
    df = df.copy()
    seen = {}
    safe_cols = []

    for col in df.columns:
        c = str(col)
        c = re.sub(r"[\[\]<>]", "_", c)
        c = re.sub(r"[^0-9A-Za-z_]+", "_", c)
        c = re.sub(r"_+", "_", c).strip("_")
        if not c:
            c = "feature"

        if c in seen:
            seen[c] += 1
            c = f"{c}_{seen[c]}"
        else:
            seen[c] = 0

        safe_cols.append(c)

    df.columns = safe_cols
    return df


def build_features(df_target, df_exog=None, lookback=28):
    feat = pd.DataFrame(index=df_target.index)
    feat["y"] = df_target.iloc[:, 0]

    for lag in range(1, lookback + 1):
        feat[f"lag_{lag}"] = feat["y"].shift(lag)

    if df_exog is not None and not df_exog.empty:
        for col in df_exog.columns:
            feat[col] = df_exog[col]

    feat = feat.dropna()
    X = feat.drop(columns=["y"])
    y = feat["y"]
    return X, y

In [ ]:
def xgboost_forecast(dataset, exo_dataset, product, product_sign, price_type, feature_nums, lookback, output_folder):
    logger.info(f"Processing {product_sign} {price_type} for {product} | features={feature_nums} | lookback={lookback}...")

    dataset = dataset.loc[dataset["PRODUCT"] == product].copy()
    if dataset.empty:
        logger.warning(f"No data found for PRODUCT={product}")
        return None

    capacity_price_column = f"{product_sign}_GERMANY_{price_type}_CAPACITY_PRICE_[(EUR/MW)/h]"
    if capacity_price_column not in dataset.columns:
        logger.warning(f"Column '{capacity_price_column}' not found")
        return None

    y_raw = dataset[[capacity_price_column]].dropna().copy()
    df_init = y_raw.copy()

    # --- Pre-processing (LSTM-style) ---
    # Step 1: log1p transform (safe for zeros)
    y_log = np.log1p(y_raw)

    # Step 2: MinMaxScaler on log-transformed target
    scaler_y = MinMaxScaler(feature_range=(0, 1))
    y_scaled = pd.DataFrame(
        scaler_y.fit_transform(y_log),
        index=y_log.index,
        columns=y_log.columns
    )

    exo_product = None
    exo_scaled = None
    if exo_dataset is not None:
        exo_product = exo_dataset.loc[exo_dataset["PRODUCT"] == product].drop(columns=["PRODUCT"], errors="ignore")
        common_idx = y_scaled.index.intersection(exo_product.index)
        y_scaled = y_scaled.loc[common_idx]
        exo_product = exo_product.loc[common_idx]

        # Scale exogenous features separately (like scaler2 in lstm_feature)
        scaler_exog = MinMaxScaler(feature_range=(0, 1))
        exo_scaled = pd.DataFrame(
            scaler_exog.fit_transform(exo_product),
            index=exo_product.index,
            columns=exo_product.columns
        )

    X, y = build_features(y_scaled, exo_scaled, lookback=lookback)
    X = make_xgb_safe_columns(X)

    n = len(X)
    train_end = int(n * 0.70)
    valid_end = int(n * 0.85)

    X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
    X_valid, y_valid = X.iloc[train_end:valid_end], y.iloc[train_end:valid_end]
    X_test, y_test = X.iloc[valid_end:], y.iloc[valid_end:]

    # --- Optuna hyperparameter search (TimeSeriesSplit on train+val) ---
    X_cv = pd.concat([X_train, X_valid])
    y_cv = pd.concat([y_train, y_valid])

    def optuna_objective(trial):
        params = {
            "objective": "reg:squarederror",
            "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "random_state": 42,
            "n_jobs": 1,
        }
        tscv = TimeSeriesSplit(n_splits=5)
        cv_scores = []
        for train_idx, val_idx in tscv.split(X_cv):
            X_fold_train, X_fold_val = X_cv.iloc[train_idx], X_cv.iloc[val_idx]
            y_fold_train, y_fold_val = y_cv.iloc[train_idx], y_cv.iloc[val_idx]
            m = XGBRegressor(**params)
            m.fit(X_fold_train, y_fold_train, eval_set=[(X_fold_val, y_fold_val)], verbose=False)
            preds = m.predict(X_fold_val)
            cv_scores.append(np.mean((preds - y_fold_val.values) ** 2))
        return np.mean(cv_scores)

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=42),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=3),
        study_name=f"xgboost_{product}_{product_sign}_{price_type}"
    )
    study.optimize(optuna_objective, n_trials=30)

    best_params = study.best_params
    best_params.update({"objective": "reg:squarederror", "random_state": 42, "n_jobs": 1})
    logger.info(f"Best params for {product} {product_sign} {price_type}: {best_params}")

    model = XGBRegressor(**best_params)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

    valid_pred_scaled = model.predict(X_valid)
    test_pred_scaled  = model.predict(X_test)

    # --- Inverse transform (LSTM-style) ---
    # Step 1: inverse MinMaxScaler
    valid_pred_log = scaler_y.inverse_transform(valid_pred_scaled.reshape(-1, 1)).flatten()
    test_pred_log  = scaler_y.inverse_transform(test_pred_scaled.reshape(-1, 1)).flatten()

    # Step 2: inverse log1p
    valid_pred_inv = np.expm1(valid_pred_log)
    test_pred_inv  = np.expm1(test_pred_log)

    valid_dates = X_valid.index
    test_dates  = X_test.index
    valid_actual = df_init.loc[valid_dates].iloc[:, 0].values
    test_actual  = df_init.loc[test_dates].iloc[:, 0].values

    valid_results_df = pd.DataFrame({
        "DATE": valid_dates,
        "ACTUAL_VALUE": valid_actual,
        "D+1": valid_pred_inv
    })
    test_results_df = pd.DataFrame({
        "DATE": test_dates,
        "ACTUAL_VALUE": test_actual,
        "D+1": test_pred_inv
    })

    results_df = pd.concat([valid_results_df, test_results_df], ignore_index=True)

    feature_nums_str = "_".join(map(str, feature_nums))
    results_df.to_csv(
        output_folder / f"xgboost_exog_{feature_nums_str}_model_results_{product}_{product_sign}_{price_type}.csv",
        index=False
    )

    overview_path = output_folder / f"xgboost_exog_{feature_nums_str}_models_overview.csv"
    overview_df = pd.read_csv(overview_path) if os.path.exists(overview_path) else pd.DataFrame()

    new_row = {
        "PRODUCT": product,
        "PRODUCT_SIGN": product_sign,
        "PRICE_TYPE": price_type,
        "LOOKBACK": lookback,
        "EXOGENOUS_FACTORS": ", ".join(exo_product.columns) if exo_product is not None else "",
        "NUM_FEATURES": 0 if exo_product is None else len(exo_product.columns),
        "FEATURE_NUMBERS": feature_nums_str,
        "N_ESTIMATORS": best_params["n_estimators"],
        "LEARNING_RATE": best_params["learning_rate"],
        "MAX_DEPTH": best_params["max_depth"],
        "MIN_CHILD_WEIGHT": best_params["min_child_weight"],
        "SUBSAMPLE": best_params["subsample"],
        "COLSAMPLE_BYTREE": best_params["colsample_bytree"],
        "REG_ALPHA": best_params["reg_alpha"],
        "REG_LAMBDA": best_params["reg_lambda"],
    }

    overview_df = pd.concat([overview_df, pd.DataFrame([new_row])], ignore_index=True)
    overview_df = overview_df.sort_values(by=["PRODUCT", "PRODUCT_SIGN", "PRICE_TYPE"])
    overview_df.to_csv(overview_path, index=False)

    logger.info(f"Done: {product} {product_sign} {price_type} (lookback={lookback})")
    return True

In [ ]:
# Define products, signs, and price types
products = ["00_04", "04_08", "08_12", "12_16", "16_20", "20_24"]
product_signs = ["NEG", "POS"]
price_types = ["AVERAGE", "MARGINAL"]
LOOKBACKS = [7]

# Read core dataset
dataset_path = BASE_INPUT_PATH / "processed_afrr_data.csv"
processed_afrr_data = pd.read_csv(dataset_path, parse_dates=["DATE"], index_col=["DATE"])

# Feature set selection
feature_nums = [4]
exog_factors = {
    "feature_1": "FCR_GERMANY_SETTLEMENTCAPACITY_PRICE_[EUR/MW]",
    "feature_2": "CUMULATED_CAPACITY_MORE_EXPENSIVE_THAN_ELECTRICITY_PRICE_EXCESS_CAPACITY",
    "feature_3": "RES_SHARE",
    "feature_4": "EXESSIVE_AVAILABLE_CAPACITY_UNTIL_PRICE_LIMIT_9999_€/MWh",
    "feature_5": "CLEAN_SPREAD_OF_GAS"
}
selected_exog_features = [exog_factors[f"feature_{n}"] for n in feature_nums]

exo_dataset_path = BASE_INPUT_PATH / "selected_exogenous_factors_20210101_20250228.csv"
exogenous_factor_data = pd.read_csv(exo_dataset_path, parse_dates=["DATE"], index_col=["DATE"])
exogenous_factor_data = exogenous_factor_data[["PRODUCT"] + selected_exog_features].copy()

# Align date index
common_dates = processed_afrr_data.index.intersection(exogenous_factor_data.index)
processed_afrr_data    = processed_afrr_data.loc[common_dates]
exogenous_factor_data  = exogenous_factor_data.loc[common_dates]

for lookback in LOOKBACKS:
    feature_nums_str = "_".join(map(str, feature_nums))
    output_folder = BASE_OUTPUT_PATH / f"xgboost_exog_{feature_nums_str}_model_output"
    output_folder.mkdir(parents=True, exist_ok=True)

    model_results = Parallel(n_jobs=1)(
        delayed(xgboost_forecast)(
            processed_afrr_data,
            exogenous_factor_data,
            product, product_sign, price_type,
            feature_nums, lookback,
            output_folder
        )
        for product in products
        for product_sign in product_signs
        for price_type in price_types
    )